**실행 설정**

입력은 `config/paths.local.json` 또는 `SLC7A5_DATA_ROOT`로 지정합니다. 결과는 이 저장소의 `results/`에 기록합니다. 입력/실행 설명은 저장소 루트 README를 참고하세요. 이 노트북에는 기존 탐색/대체 Figure 셀이 포함되어 있으며 전체 Figure 재현 검증은 별도 단계입니다.

In [ ]:
# Repository paths; inputs are read-only, new files stay in results/.
from pathlib import Path
import os
import sys
_start = Path(os.environ.get("SLC7A5_REPO_ROOT", str(Path.cwd()))).resolve()
_repo = next((p for p in [_start, *_start.parents] if (p / "slc7a5_paths.py").is_file()), None)
if _repo is None:
    raise RuntimeError("Open this notebook inside the repository or set SLC7A5_REPO_ROOT.")
if str(_repo) not in sys.path:
    sys.path.insert(0, str(_repo))
from slc7a5_paths import artifact_path, dataset_path, input_path, output_path, output_dir, setup_notebook
setup_notebook('nucseq_gsea')


In [ ]:
# Import package
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc



In [ ]:
# adata 경로를 실제 파일 경로로 수정하세요
adata = sc.read(artifact_path('50_Public_GEO_analysis/analysis/Data/Liver_DB_nuc_stage.h5ad'))
print(f"전체 세포 수: {adata.n_obs:,}, 유전자 수: {adata.n_vars:,}")
adata

In [ ]:
adata

In [ ]:
sc.pl.umap(adata, color=['celltype'])

In [ ]:
import gseapy as gp                                                                                                                       

gene = 'SLC7A5'                                                                                                                           

# ── 1. SLC7A5 pos/neg 레이블 부착 ──────────────────────────────                                                                         
gene_idx = list(adata.var_names).index(gene)
expr_vec = adata.X[:, gene_idx]                                                                                                           
if hasattr(expr_vec, 'toarray'):
    expr_vec = expr_vec.toarray().flatten()                                                                                               
else:                                                                                                                                     
    expr_vec = np.array(expr_vec).flatten()

adata.obs['SLC7A5_group'] = np.where(expr_vec > 0, 'pos', 'neg')

# ── 2. 3개 celltype 필터 ───────────────────────────────────────                                                                         
# ※ Hepatocyte가 'celltype' 컬럼에 없으면 아래에서 확인
# print(adata.obs['celltypist_liver'].unique())                                                                                           
target_celltypes = ['Hepatocyte', 'Stellete', 'Macrophage']                                                                                  
adata_sub = adata[adata.obs['celltype'].isin(target_celltypes)].copy()       

In [ ]:
for ct in target_celltypes:
      sub = adata_sub[adata_sub.obs['celltype'] == ct]
      counts = sub.obs['SLC7A5_group'].value_counts()                                                                                       
      total = len(sub)                                                                                                                      
      print(f"\n[{ct}] total={total}")                                                                                                      
      for grp in ['pos', 'neg']:                                                                                                            
          n = counts.get(grp, 0)
          print(f"  {grp}: {n} ({n/total*100:.1f}%)") 

In [ ]:
# ── 3. celltype별 GSEA ─────────────────────────────────────────                                                                         
gene_sets = 'MSigDB_Hallmark_2020'   # 또는 로컬 .gmt 파일 경로
results   = {}                                                                                                                            

for ct in target_celltypes:                                                                                                               
    sub = adata_sub[adata_sub.obs['celltype'] == ct].copy()

    grp_counts = sub.obs['SLC7A5_group'].value_counts()                                                                                   
    if 'pos' not in grp_counts or 'neg' not in grp_counts or grp_counts.min() < 3:
        print(f"[{ct}] pos/neg 세포 부족, skip")                                                                                          
        continue                                                                                                                          

    # pos vs neg Wilcoxon → ranked gene list                                                                                              
    sc.tl.rank_genes_groups(
        sub,                                                                                                                              
        groupby='SLC7A5_group',
        groups=['pos'],
        reference='neg',
        method='wilcoxon',
        use_raw=False
    )
    deg = sc.get.rank_genes_groups_df(sub, group='pos')

    ranked = (deg[['names', 'scores']]                                                                                                    
              .dropna()                                                                                                                   
              .drop_duplicates('names')                                                                                                   
              .sort_values('scores', ascending=False)
              .set_index('names')['scores'])

    # prerank GSEA                                                                                                                        
    pre_res = gp.prerank(
        rnk=ranked,                                                                                                                       
        gene_sets=gene_sets,
        threads=4,
        min_size=15,
        max_size=500,
        permutation_num=1000,
        outdir=None,                                                                                                                      
        seed=42,
        verbose=False                                                                                                                     
    )           

    results[ct] = pre_res
    print(f"[{ct}] 완료: {len(pre_res.res2d)} pathways")

# ── 4. 결과 요약 출력 ─────────────────────────────────────────
for ct, pre_res in results.items():                                                                                                       
    print(f"\n=== {ct} | SLC7A5 pos vs neg | top NES ===")
    print(pre_res.res2d.sort_values('NES', ascending=False)                                                                               
                       .head(10)[['Term', 'NES', 'NOM p-val', 'FDR q-val']]) 

In [ ]:
keywords = ['gly', 'fatty acid', 'lipid', 'cycle', 'hif', 'tor','pha']

for ct, pre_res in results.items():
    df = pre_res.res2d.copy()

    mask = df['Term'].str.lower().str.contains('|'.join(keywords))
    matched = df[mask].sort_values('NES', ascending=False)

    print(f"\n=== {ct} | keyword match ===")
    if matched.empty:
        print("No matching pathways found")
    else:
        print(matched[['Term', 'NES', 'NOM p-val', 'FDR q-val']])

In [ ]:
# pos, neg 각각 모든 유전자 대비 GSEA
results_sep = {}  # results_sep[ct]['pos'], results_sep[ct]['neg']                                                                        

for ct in target_celltypes:                                                                                                               
    sub = adata_sub[adata_sub.obs['celltype'] == ct].copy()                                                                               
    results_sep[ct] = {}                                                                                                                  

    for grp, ref in [('pos', 'neg'), ('neg', 'pos')]:                                                                                     
        grp_cells = sub.obs['SLC7A5_group'] == grp
        if grp_cells.sum() < 3:                                                                                                           
            continue

        sc.tl.rank_genes_groups(
            sub,
            groupby='SLC7A5_group',                                                                                                       
            groups=[grp],
            reference=ref,                                                                                                                
            method='wilcoxon',
            use_raw=False
        )                                                                                                                                 
        deg = sc.get.rank_genes_groups_df(sub, group=grp)
        ranked = (deg[['names', 'scores']]                                                                                                
                  .dropna()                                                                                                               
                  .drop_duplicates('names')                                                                                               
                  .sort_values('scores', ascending=False)                                                                                 
                  .set_index('names')['scores'])                                                                                          

        pre_res = gp.prerank(                                                                                                             
            rnk=ranked,
            gene_sets=gene_sets,                                                                                                          
            threads=4,
            min_size=15,                                                                                                                  
            max_size=500,                                                                                                                 
            permutation_num=1000,
            outdir=None,                                                                                                                  
            seed=42,
            verbose=False                                                                                                                 
        )
        results_sep[ct][grp] = pre_res

In [ ]:
# 관심 term 키워드 (대소문자 무관하게 매칭)                                                                                               
keywords = [                                                                                                                              
    'MTOR', 'HIF1', 'AUTOPHAGY',
    'GLYCOLYSIS', 'GLUCONEOGENESIS', 'TCA', 'OXIDATIVE PHOSPHORYLATION',                                                                  
    'Fatty Acid'                                                                                                              
]                                                                                                                                         

def match_terms(res_df, keywords):                                                                                                        
    matched = []
    for kw in keywords:                                                                                                                   
        hits = res_df[res_df['Term'].str.upper().str.contains(kw)]
        if not hits.empty:                                                                                                                
            matched.append(hits.iloc[0]['Term'])
    return matched     

In [ ]:
# ── 1. 개별 enrichment plot ───────────────────────────────────                                                                          
for ct, grp_dict in results_sep.items():                                                                                                  
    for grp, pre_res in grp_dict.items():
        matched = match_terms(pre_res.res2d, keywords)                                                                                    
        print(f"[{ct} | {grp}] matched: {matched}")

        for term in matched:
            row       = pre_res.res2d[pre_res.res2d['Term'] == term].iloc[0]
            term_data = pre_res.results.get(term, {})                                                                                     
            ranking   = term_data.get('RES', None)
            if ranking is None:                                                                                                           
                continue

            hit_idx = (term_data.get('hit_indices')
                       or term_data.get('hits')
                       or term_data.get('hit_index')
                       or [])                                                                                                             

            color = '#3498DB' if grp == 'pos' else '#E74C3C'                                                                              

            fig, ax = plt.subplots(figsize=(5, 3))                                                                                        
            ax.plot(ranking, color=color, lw=1.5)
            ax.axhline(0, color='black', lw=0.5, linestyle='--')                                                                          
            ax.axvline(np.argmax(np.abs(ranking)), color='red', lw=0.8, linestyle='--')
            for idx in hit_idx:                                                                                                           
                ax.axvline(idx, color='grey', lw=0.3, alpha=0.4)

            nes = row['NES']
            fdr = row['FDR q-val']                                                                                                        
            ax.set_title(f"{ct} | {grp} | {term}\nNES={nes:.2f}, FDR={fdr:.3f}", fontsize=8)
            ax.set_xlabel('Rank', fontsize=7)                                                                                             
            ax.set_ylabel('Enrichment Score', fontsize=7)
            ax.tick_params(labelsize=6)                                                                                                   
            plt.tight_layout()
            #plt.savefig(f'GSEA_{ct}_{term[:30]}.pdf', bbox_inches='tight')
            plt.show()                                                                                                                  

In [ ]:
def sig_star(p):                                                                                                                          
    if np.isnan(p): return ''                                                                                                             
    if p < 0.001:   return '***'                                                                                                          
    if p < 0.01:    return '**'                                                                                                           
    if p < 0.05:    return '*'
    return ''

def fmt_pval(p):
    if np.isnan(p): return 'NA'
    return f'{p:.2e}' if p < 0.001 else f'{p:.3f}'

# all_terms 수집 — ct → grp → pre_res 구조로                                                                                              
all_terms = set()
for ct, grp_dict in results_sep.items():                                                                                                  
    for grp, pre_res in grp_dict.items():
        all_terms.update(match_terms(pre_res.res2d, keywords))                                                                            
all_terms = sorted(all_terms)

celltypes_list = list(results_sep.keys())
linestyles = ['-', '--', ':']
colors     = {'pos': '#3498DB', 'neg': '#E74C3C'}                                                                                         

n_pw = len(all_terms)                                                                                                                     
fig, axes = plt.subplots(1, n_pw, figsize=(n_pw * 5, 4.5), constrained_layout=True)
if n_pw == 1:                                                                                                                             
    axes = [axes]

for ax, term in zip(axes, all_terms):
    legend_handles = []                                                                                                                   

    for ci, ct in enumerate(celltypes_list):
        ls = linestyles[ci % len(linestyles)]

        for grp in ['pos', 'neg']:
            if grp not in results_sep[ct]:                                                                                                
                continue

            pre_res   = results_sep[ct][grp]
            term_data = pre_res.results.get(term, {})                                                                                     
            es_curve  = term_data.get('RES', None)
            if es_curve is None or len(es_curve) == 0:
                continue                                                                                                                  

            row = pre_res.res2d[pre_res.res2d['Term'] == term]                                                                            
            if row.empty:
                continue

            nes  = row.iloc[0]['NES']                                                                                                     
            fdr  = row.iloc[0]['FDR q-val']

            color = colors[grp]
            x     = np.linspace(0, 1, len(es_curve))
            ax.plot(x, es_curve, color=color, lw=2.0, alpha=0.85, ls=ls)                                                                  

            star = sig_star(fdr)                                                                                                          
            lbl  = f"{ct} ({grp})  NES={nes:.2f}  FDR={fmt_pval(fdr)}{' '+star if star else ''}"
            legend_handles.append(                                                                                                        
                plt.Line2D([0], [0], color=color, lw=2, ls=ls, label=lbl)
            )                                                                                                                             

    ax.axhline(0, color='grey', lw=0.8, ls='--')                                                                                          
    ax.set_title(term, fontsize=10, fontweight='bold')
    ax.set_xlabel('Gene rank (normalized)', fontsize=8)                                                                                   
    ax.set_ylabel('Running ES', fontsize=8)
    ax.spines['top'].set_visible(False)                                                                                                   
    ax.spines['right'].set_visible(False)
    ax.legend(handles=legend_handles, fontsize=7, framealpha=0.35,                                                                        
              loc='best', title='line=celltype / color=group', title_fontsize=7)

fig.suptitle(   
    'GSEA ES Curves — SLC7A5 pos (blue) vs neg (red) per celltype\n'                                                                      
    '(line style = celltype)',
    fontsize=12, fontweight='bold'                                                                                                        
)
#plt.savefig('GSEA_ES_curves_pos_neg.pdf', bbox_inches='tight', dpi=300)                                                                   
plt.show()  

In [ ]:
# index를 "Myeloid_pos", "Myeloid_neg" 형태로                                                                                             
all_terms = set()                                                                                                                         
for ct, grp_dict in results_sep.items():                                                                                                  
    for grp, pre_res in grp_dict.items():                                                                                                 
        all_terms.update(match_terms(pre_res.res2d, keywords))
all_terms = sorted(all_terms)

row_index = [f"{ct}_{grp}" for ct in results_sep for grp in ['pos', 'neg']
             if grp in results_sep[ct]]                                                                                                   

nes_mat = pd.DataFrame(index=row_index, columns=all_terms, dtype=float)
fdr_mat = pd.DataFrame(index=row_index, columns=all_terms, dtype=float)

for ct, grp_dict in results_sep.items():
    for grp, pre_res in grp_dict.items():                                                                                                 
        idx = f"{ct}_{grp}"
        for term in all_terms:                                                                                                            
            row = pre_res.res2d[pre_res.res2d['Term'] == term]
            if not row.empty:                                                                                                             
                nes_mat.loc[idx, term] = row.iloc[0]['NES']
                fdr_mat.loc[idx, term] = row.iloc[0]['FDR q-val']                                                                         

nes_mat  = nes_mat.astype(float)                                                                                                          
fdr_mat  = fdr_mat.astype(float)
sig_size = -np.log10(fdr_mat.fillna(1) + 1e-10)                                                                                           

# pos=파랑, neg=빨강 (행 이름 기준)                                                                                                       
def row_color(idx_name, nes):
    grp = idx_name.split('_')[-1]                                                                                                         
    return '#3498DB' if grp == 'pos' else '#E74C3C'                                                                                       

fig, ax = plt.subplots(figsize=(len(all_terms) * 0.9 + 1, len(row_index) * 0.8 + 1))                                                      

for i, idx in enumerate(nes_mat.index):                                                                                                   
    for j, term in enumerate(nes_mat.columns):
        nes = nes_mat.loc[idx, term]                                                                                                      
        sz  = max(sig_size.loc[idx, term] * 100, 5)
        fdr = fdr_mat.loc[idx, term]
        if np.isnan(nes):                                                                                                                 
            continue
        color = row_color(idx, nes)                                                                                                       
        ax.scatter(j, i, s=sz, c=color, alpha=0.8, edgecolors='k', linewidths=0.4)
        if not np.isnan(fdr) and fdr < 0.05:                                                                                              
            ax.text(j, i, '*', ha='center', va='center', fontsize=9,
                    fontweight='bold', color='white')                                                                                     

ax.set_xticks(range(len(all_terms)))
ax.set_xticklabels(all_terms, rotation=40, ha='right', fontsize=8)                                                                        
ax.set_yticks(range(len(row_index)))
ax.set_yticklabels(nes_mat.index, fontsize=9)                                                                                             
ax.set_xlim(-0.5, len(all_terms) - 0.5)
ax.set_ylim(-0.5, len(row_index) - 0.5)                                                                                                   
ax.grid(True, linestyle='--', alpha=0.3)                                                                                                  
ax.set_title('SLC7A5 pos (blue) vs neg (red) | GSEA NES\n(size=-log10 FDR, *=FDR<0.05)', fontsize=10)

for label, color in [('pos group', '#3498DB'), ('neg group', '#E74C3C')]:                                                                 
    ax.scatter([], [], c=color, s=80, label=label, alpha=0.8)                                                                             
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1), fontsize=7)                                                                        

plt.tight_layout()                                                                                                                        
#plt.savefig('GSEA_bubble_summary.pdf', bbox_inches='tight')
plt.show()    

In [ ]:
import scanpy as sc
# KEGG용 results_sep 별도 생성
gene_sets_kegg = 'KEGG_2021_Human'                                                                                                        
results_sep_kegg = {}                                                                                                                     

for ct in target_celltypes:                                                                                                               
    sub = adata_sub[adata_sub.obs['celltype'] == ct].copy()                                                                             
    results_sep_kegg[ct] = {}                                                                                                             

    for grp, ref in [('pos', 'neg'), ('neg', 'pos')]:                                                                                     
        if (sub.obs['SLC7A5_group'] == grp).sum() < 3:                                                                                  
            continue                                                                                                                      
        sc.tl.rank_genes_groups(sub, groupby='SLC7A5_group',                                                                            
                                groups=[grp], reference=ref,                                                                              
                                method='wilcoxon', use_raw=False)                                                                       
        deg = sc.get.rank_genes_groups_df(sub, group=grp)                                                                                 
        ranked = (deg[['names', 'scores']].dropna()                                                                                       
                  .drop_duplicates('names')                                                                                               
                  .sort_values('scores', ascending=False)                                                                                 
                  .set_index('names')['scores'])                                                                                          
        pre_res = gp.prerank(rnk=ranked, gene_sets=gene_sets_kegg,                                                                      
                             threads=4, min_size=15, max_size=500,                                                                        
                             permutation_num=1000, outdir=None,                                                                           
                             seed=42, verbose=False)                                                                                      
        results_sep_kegg[ct][grp] = pre_res    

In [ ]:
keywords = ['pi3k','mtor', 'hif', 'auto',
      'glycol', 'tca', 'oxidative',                                                                  
      'fatty' ]

for ct, grp_dict in results_sep_kegg.items():                                                                                             
      for grp, pre_res in grp_dict.items():                                                                                                 
          df = pre_res.res2d.copy()                                                                                                         
          mask = df['Term'].str.lower().str.contains('|'.join(keywords))                                                                    
          matched = df[mask].sort_values('NES', ascending=False)                                                                            
          print(f"\n=== {ct} | {grp} | keyword match ===")
          if matched.empty:                                                                                                                 
              print("No matching pathways found")
          else:                                                                                                                             
              print(matched[['Term', 'NES', 'NOM p-val', 'FDR q-val']])

In [ ]:
# 관심 term 키워드 (대소문자 무관하게 매칭)                                                                                               
keywords = ['pi3k','mtor', 'hif', 'auto',
    'glycol', 'tca', 'oxidative',                                                                  
    'fatty' ]                                                                                                                                      

def match_terms(res_df, keywords):                                                                                                        
    matched = []
    for kw in keywords:                                                                                                                   
        hits = res_df[res_df['Term'].str.lower().str.contains(kw.lower())]
        if not hits.empty:                                                                                                                
            matched.append(hits.iloc[0]['Term'])
    return matched     

In [ ]:
# ── 1. 개별 enrichment plot ───────────────────────────────────                                                                          
for ct, grp_dict in results_sep_kegg.items():                                                                                                  
    for grp, pre_res in grp_dict.items():
        matched = match_terms(pre_res.res2d, keywords)                                                                                    
        print(f"[{ct} | {grp}] matched: {matched}")

        for term in matched:
            row       = pre_res.res2d[pre_res.res2d['Term'] == term].iloc[0]
            term_data = pre_res.results.get(term, {})                                                                                     
            ranking   = term_data.get('RES', None)
            if ranking is None:                                                                                                           
                continue

            hit_idx = (term_data.get('hit_indices')
                       or term_data.get('hits')
                       or term_data.get('hit_index')
                       or [])                                                                                                             

            color = '#3498DB' if grp == 'pos' else '#E74C3C'                                                                              

            fig, ax = plt.subplots(figsize=(5, 3))                                                                                        
            ax.plot(ranking, color=color, lw=1.5)
            ax.axhline(0, color='black', lw=0.5, linestyle='--')                                                                          
            ax.axvline(np.argmax(np.abs(ranking)), color='red', lw=0.8, linestyle='--')
            for idx in hit_idx:                                                                                                           
                ax.axvline(idx, color='grey', lw=0.3, alpha=0.4)

            nes = row['NES']
            fdr = row['FDR q-val']                                                                                                        
            ax.set_title(f"{ct} | {grp} | {term}\nNES={nes:.2f}, FDR={fdr:.3f}", fontsize=8)
            ax.set_xlabel('Rank', fontsize=7)                                                                                             
            ax.set_ylabel('Enrichment Score', fontsize=7)
            ax.tick_params(labelsize=6)                                                                                                   
            plt.tight_layout()
            #plt.savefig(f'GSEA_{ct}_{term[:30]}.pdf', bbox_inches='tight')
            plt.show()                                                                                                                  

In [ ]:
def sig_star(p):                                                                                                                          
    if np.isnan(p): return ''                                                                                                             
    if p < 0.001:   return '***'                                                                                                          
    if p < 0.01:    return '**'                                                                                                           
    if p < 0.05:    return '*'
    return ''

def fmt_pval(p):
    if np.isnan(p): return 'NA'
    return f'{p:.2e}' if p < 0.001 else f'{p:.3f}'

# all_terms 수집 — ct → grp → pre_res 구조로                                                                                              
all_terms = set()
for ct, grp_dict in results_sep_kegg.items():                                                                                                  
    for grp, pre_res in grp_dict.items():
        all_terms.update(match_terms(pre_res.res2d, keywords))                                                                            
all_terms = sorted(all_terms)

celltypes_list = list(results_sep_kegg.keys())
linestyles = ['-', '--', ':']
colors     = {'pos': '#3498DB', 'neg': '#E74C3C'}                                                                                         

n_pw = len(all_terms)                                                                                                                     
fig, axes = plt.subplots(1, n_pw, figsize=(n_pw * 5, 4.5), constrained_layout=True)
if n_pw == 1:                                                                                                                             
    axes = [axes]

for ax, term in zip(axes, all_terms):
    legend_handles = []                                                                                                                   

    for ci, ct in enumerate(celltypes_list):
        ls = linestyles[ci % len(linestyles)]

        for grp in ['pos', 'neg']:
            if grp not in results_sep_kegg[ct]:                                                                                                
                continue

            pre_res   = results_sep_kegg[ct][grp]
            term_data = pre_res.results.get(term, {})                                                                                     
            es_curve  = term_data.get('RES', None)
            if es_curve is None or len(es_curve) == 0:
                continue                                                                                                                  

            row = pre_res.res2d[pre_res.res2d['Term'] == term]                                                                            
            if row.empty:
                continue

            nes  = row.iloc[0]['NES']                                                                                                     
            fdr  = row.iloc[0]['FDR q-val']

            color = colors[grp]
            x     = np.linspace(0, 1, len(es_curve))
            ax.plot(x, es_curve, color=color, lw=2.0, alpha=0.85, ls=ls)                                                                  

            star = sig_star(fdr)                                                                                                          
            lbl  = f"{ct} ({grp})  NES={nes:.2f}  FDR={fmt_pval(fdr)}{' '+star if star else ''}"
            legend_handles.append(                                                                                                        
                plt.Line2D([0], [0], color=color, lw=2, ls=ls, label=lbl)
            )                                                                                                                             

    ax.axhline(0, color='grey', lw=0.8, ls='--')                                                                                          
    ax.set_title(term, fontsize=10, fontweight='bold')
    ax.set_xlabel('Gene rank (normalized)', fontsize=8)                                                                                   
    ax.set_ylabel('Running ES', fontsize=8)
    ax.spines['top'].set_visible(False)                                                                                                   
    ax.spines['right'].set_visible(False)
    ax.legend(handles=legend_handles, fontsize=7, framealpha=0.35,                                                                        
              loc='best', title='line=celltype / color=group', title_fontsize=7)

fig.suptitle(   
    'GSEA ES Curves — SLC7A5 pos (blue) vs neg (red) per celltype\n'                                                                      
    '(line style = celltype)',
    fontsize=12, fontweight='bold'                                                                                                        
)
plt.savefig(output_path('02_SLC7A5_mono_Journal/05_Figure/Nuc_KEGGGSEA_ES_curves_pos_neg.pdf'), bbox_inches='tight', dpi=300)                                                                   
plt.show()  